# 04 · LoRA, assembled

**The whole chain, in one breath:**
- 01: a layer is `output = W @ x`.
- 02: a low-rank matrix can be stored cheaply as **`B · A`** (with `r` directions).
- 03: when fine-tuning, the change **`ΔW` is usually low rank**, and we can keep `W` frozen.

Put those together and you get **LoRA**: don't learn the whole change `ΔW`. Freeze `W`,
and learn a small `B` and `A` so that `ΔW = B · A`:

$$W_{\text{effective}} = W + B A \qquad (\text{we'll add one knob, } \alpha, \text{ shortly})$$

In [ ]:
import numpy as np
np.random.seed(0)

## Step 1 — Why we bother: the parameter savings

This is the entire reason LoRA exists. Take one realistic weight matrix of size
1536×1536. Changing **all** its knobs (a full `ΔW`) means learning 1536×1536 numbers.
Storing the change as `B · A` with a small `r` means learning only `r × (1536 + 1536)`.

In [ ]:
d = k = 1536
full = d * k
print(f"full ΔW (change every knob): {full:,} numbers\n")
print(f"{'r':>4} {'LoRA: B and A together':>24} {'fraction of full':>18}")
for r in (4, 8, 16, 32, 64):
    lora = r * (d + k)
    print(f"{r:>4} {lora:>24,} {lora/full:>17.1%}")

At `r = 8` you train **about 1%** of what a full change would cost — for *one* matrix, and
the saving repeats across every layer. That's the headline.

## Step 2 — The zero-start trick

We initialize `B` to **all zeros**. Then `B · A = 0`, so `W_effective = W + 0 = W`.
Meaning: at the very first step, **the LoRA model IS the pretrained model** — we haven't
damaged anything. It only departs from the base model as `B` learns.

In [ ]:
d, k, r = 8, 5, 2
W = np.random.randn(d, k)
x = np.random.randn(k)
A = np.random.randn(r, k)
B = np.zeros((d, r))           # <-- start B at zero

print("B @ A at the start :", np.abs(B @ A).max(), "(all zeros)")
print("(W + B@A)x  minus  W x :", np.abs((W + B @ A) @ x - W @ x).max(),
      " -> identical to the base model")

## Step 3 — Finally, `α` (alpha): the volume knob

`B` and `A` learn the *shape* of the change. We also want a simple **dial for how
strongly** to apply that change. That dial is **`α` (alpha)**, and LoRA applies it as
`α / r`:

$$W_{\text{effective}} = W + \frac{\alpha}{r}\, B A$$

Bigger `α` → the change is applied more strongly. Watch a fixed `B·A` get louder as we
turn `α` up:

In [ ]:
d, k, r = 64, 64, 8
B = np.random.randn(d, r)
A = np.random.randn(r, k)
raw = np.linalg.norm(B @ A)          # the natural size of the change
print(f"raw size of B@A: {raw:.1f}\n")
print(f"{'alpha':>6} {'alpha/r':>8} {'size of applied change':>24}")
for alpha in (4, 8, 16, 32):
    print(f"{alpha:>6} {alpha/r:>8.2f} {(alpha/r) * raw:>24.1f}")

**Why divide by `r`?** Because if you later use more directions (bigger `r`), the raw
`B·A` tends to get bigger too. Dividing by `r` keeps the applied strength in a similar
range, so the *same* `α` keeps working as you change `r`. That's exactly what lets us
compare `r = 4, 8, 16, 32, 64` fairly later. (Common choices: `α = r` → dial 1.0, or
`α = 2r` → dial 2.0.)

## Step 4 — But does a small `r` actually capture the change?

Last honest question. Suppose the *ideal* change really needs only **5** directions. Can a
rank-`r` `B·A` capture it? Yes — and there's a theorem (Eckart–Young) saying `B·A` can
capture the **best possible** rank-`r` chunk of any matrix. Let's watch how much of a
"true rank-5" change we capture as we raise `r`:

In [ ]:
d = k = 64
# an 'ideal change' that genuinely needs only 5 directions (+ a little noise):
ideal = np.random.randn(d, 5) @ np.random.randn(5, k) + 0.02 * np.random.randn(d, k)

U, S, Vt = np.linalg.svd(ideal, full_matrices=False)
total = np.linalg.norm(ideal)
print("strength of each direction (top 8):", np.round(S[:8], 2), "\n")
print(f"{'r':>4} {'fraction of the change captured':>32}")
for r in (1, 2, 5, 10):
    approx = (U[:, :r] * S[:r]) @ Vt[:r]      # the best rank-r B@A
    captured = 1 - np.linalg.norm(ideal - approx) / total
    print(f"{r:>4} {captured:>31.1%}")

Five strong directions, then basically nothing — and by `r = 5` we capture ~all of the
change. When the real change is low rank, a small LoRA captures it for almost no cost.

**The honest catch:** this worked because we *built* the change to be rank-5. Whether
**real** text-to-SQL fine-tuning changes are this compressible is something we *measure*,
not assume — by sweeping `r = 4, 8, 16, 32, 64` on a real GPU run and watching where the
accuracy stops improving.

## Recap — the whole story

| notebook | idea |
|---|---|
| 01 | a layer is `W @ x`; fine-tuning changes `W` |
| 02 | a low-rank matrix = `B · A` (with `r` directions) |
| 03 | the change `ΔW` is usually low rank; keep `W` frozen |
| 04 | so learn `B`, `A` instead → `W + (α/r)·B·A` = **LoRA** |

You now understand LoRA from the ground up — every symbol earned. **Triple BAM!**

Next step is the *outcome* evidence: run `python run.py --config lora` on Kaggle and do
the `r`-sweep to see, on real data, whether SQL fine-tuning truly lives at low rank.